In [9]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
BASE_DIR = Path(".").resolve()

DATA_PATH = BASE_DIR / "data/processed/cycle_features_with_rul.csv"
MODEL_OUT = BASE_DIR / "data/processed/modeling"

ANOMALY_PATH = MODEL_OUT / "anomalies.json"
UNCERTAINTY_PATH = MODEL_OUT / "uncertainty_estimates.json"
SURVIVAL_PATH = MODEL_OUT / "survival_risk_predictions.csv"


In [ ]:
df = pd.read_csv(DATA_PATH)

with open(ANOMALY_PATH, "r") as f:
    anomalies = json.load(f)

with open(UNCERTAINTY_PATH, "r") as f:
    uncertainty = json.load(f)

if SURVIVAL_PATH.exists():
    survival = pd.read_csv(SURVIVAL_PATH)
else:
    survival = pd.DataFrame()

print("Data shape:", df.shape)
print("Anomalies:", len(anomalies))
print("Uncertainty records:", len(uncertainty))
print("Survival risk rows:", len(survival))


In [ ]:
u_preview = pd.DataFrame(uncertainty)
s_preview = survival.copy()

df_ids = set(df["battery_id"].dropna().astype(str).unique())
u_ids = (
    set(u_preview["battery_id"].dropna().astype(str).unique())
    if "battery_id" in u_preview.columns
    else set()
)
s_ids = (
    set(s_preview["battery_id"].dropna().astype(str).unique())
    if (not s_preview.empty and "battery_id" in s_preview.columns)
    else set()
)

if s_ids:
    shared_ids = sorted(df_ids & u_ids & s_ids)
else:
    shared_ids = sorted(df_ids & u_ids)

if shared_ids:
    battery_id = shared_ids[0]
else:
    battery_id = df["battery_id"].iloc[0]

df_b = df[df["battery_id"] == battery_id]

plt.figure(figsize=(10, 4))
plt.plot(df_b["cycle_index"], df_b["capacity"], label="Capacity")
plt.xlabel("Cycle")
plt.ylabel("Capacity")
plt.title(f"Degradation Trajectory - Battery {battery_id}")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
an_df = pd.DataFrame(anomalies)

if {"battery_id", "cycle_index"}.issubset(an_df.columns):
    an_b = an_df[an_df["battery_id"] == battery_id]
    anomaly_cycles = an_b[["cycle_index"]].drop_duplicates()
else:
    anomaly_cycles = pd.DataFrame(columns=["cycle_index"])

df_merge = df_b.merge(
    anomaly_cycles,
    on="cycle_index",
    how="left",
    indicator=True
)

plt.figure(figsize=(10, 4))
plt.plot(df_b["cycle_index"], df_b["capacity"], label="Capacity")
plt.scatter(
    df_merge[df_merge["_merge"] == "both"]["cycle_index"],
    df_merge[df_merge["_merge"] == "both"]["capacity"],
    color="red",
    label="Anomalies",
    zorder=3
)

plt.xlabel("Cycle")
plt.ylabel("Capacity")
plt.title(f"Detected Anomalies - Battery {battery_id}")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
u_df = pd.DataFrame(uncertainty)
required_u_cols = {"battery_id", "cycle_index", "rul_median", "rul_lower_5", "rul_upper_95"}

if required_u_cols.issubset(u_df.columns):
    u_b = u_df[u_df["battery_id"] == battery_id].sort_values("cycle_index")
else:
    u_b = pd.DataFrame(columns=sorted(required_u_cols))

if u_b.empty:
    print(f"No uncertainty records found for battery {battery_id}.")
else:
    plt.figure(figsize=(10, 4))
    plt.plot(u_b["cycle_index"], u_b["rul_median"], label="Median RUL")
    plt.fill_between(
        u_b["cycle_index"],
        u_b["rul_lower_5"],
        u_b["rul_upper_95"],
        alpha=0.3,
        label="90% Interval"
    )
    plt.xlabel("Cycle")
    plt.ylabel("Remaining Useful Life")
    plt.title(f"RUL Uncertainty - Battery {battery_id}")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
s_df = survival.copy()
required_s_cols = {"battery_id", "cycle_index", "hazard_prob", "failure_prob_horizon"}

if required_s_cols.issubset(s_df.columns):
    s_b = s_df[s_df["battery_id"].astype(str) == str(battery_id)].sort_values("cycle_index")
else:
    s_b = pd.DataFrame(columns=sorted(required_s_cols))

if s_b.empty:
    print(f"No survival risk records found for battery {battery_id}.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

    axes[0].plot(s_b["cycle_index"], s_b["hazard_prob"], color="tab:orange", label="Hazard Probability")
    axes[0].set_ylabel("Hazard")
    axes[0].set_title(f"Survival/Hazard Risk - Battery {battery_id}")
    axes[0].grid(True)
    axes[0].legend()

    axes[1].plot(
        s_b["cycle_index"],
        s_b["failure_prob_horizon"],
        color="tab:red",
        label="Failure Probability (Horizon)"
    )
    axes[1].set_xlabel("Cycle")
    axes[1].set_ylabel("Failure Probability")
    axes[1].grid(True)
    axes[1].legend()

    plt.tight_layout()
    plt.show()
